In [2]:
import os
from pathlib import Path
import pandas as pd

In [3]:
input_folder = Path("input_files")
output_file = "Sorted.xlsx"
id_column = "ID Number"
department_column = None

In [4]:
excel_files = sorted(input_folder.glob("*.xls"))

if not excel_files:
    raise FileNotFoundError("No Excel files were found in the input_files folder.")

FileNotFoundError: No Excel files were found in the input_files folder.

In [ ]:
frames = []

for file_path in excel_files:
    df = pd.read_excel(file_path, engine="xlrd")
    df = df.copy()

    if id_column in df.columns:
        if department_column is None:
            for candidate in ["Department", "DEPARTMENT", "department", "Dept", "DEPT"]:
                if candidate in df.columns:
                    department_column = candidate
                    break

        if department_column is None:
            department_column = "Department"
            df[department_column] = "UNKNOWN"
        else:
            df[department_column] = df[department_column].fillna("UNKNOWN")

        df["Source File"] = file_path.name
        frames.append(df)
    else:
        print(f"Skipping {file_path.name}: missing '{id_column}' column")

if not frames:
    raise ValueError(f"No files with the '{id_column}' column were found.")

XLRDError: Excel xlsx file; not supported

In [ ]:
combined_df = pd.concat(frames, ignore_index=True)
combined_df[department_column] = combined_df[department_column].fillna("UNKNOWN")
combined_df[id_column] = combined_df[id_column].fillna("UNKNOWN")

combined_df[department_column] = combined_df[department_column].astype(str).str.strip()
combined_df[id_column] = combined_df[id_column].astype(str).str.strip()
combined_df.loc[combined_df[id_column].eq(""), id_column] = "UNKNOWN"
combined_df.loc[combined_df[department_column].eq(""), department_column] = "UNKNOWN"

combined_df["__group_department"] = combined_df[department_column].astype(str).str.strip().str.upper()
combined_df["__group_id"] = combined_df[id_column].astype(str).str.strip().str.upper()

combined_df = combined_df.sort_values(by=["__group_id", "__group_department"], na_position="last").reset_index(drop=True)


def safe_sheet_name(value):
    text = str(value).strip()
    if not text:
        text = "UNKNOWN"
    for char in ["/", "\\", "?", "*", "[", "]", ":"]:
        text = text.replace(char, "-")
    text = text.replace("\n", " ").replace("\r", " ")
    text = " ".join(text.split())
    return text[:31]


def make_unique_sheet_name(base_name, used_names):
    name = safe_sheet_name(base_name)
    if name not in used_names:
        return name

    suffix = 1
    while True:
        candidate = f"{name[:28]}_{suffix}"
        if candidate not in used_names:
            return candidate
        suffix += 1


grouped = []
for (id_value, department_value), group in combined_df.groupby(["__group_id", "__group_department"], dropna=False):
    grouped.append((safe_sheet_name(department_value), safe_sheet_name(id_value), group))

grouped.sort(key=lambda item: (item[0], item[1]))

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    used_sheet_names = set()
    for department_name, id_name, group in grouped:
        base_name = f"{department_name} - {id_name}"
        sheet_name = make_unique_sheet_name(base_name, used_sheet_names)
        used_sheet_names.add(sheet_name)
        group.drop(columns=["Source File", "__group_id", "__group_department"], errors="ignore").to_excel(writer, sheet_name=sheet_name, index=False)

In [ ]:
print(f"Done! Workbook saved to '{output_file}'.")
print(pd.ExcelFile(output_file).sheet_names)

Done! Workbook saved to 'Sorted.xlsx'.
['MC-TOED - CASUAL', 'MC-TOED - GIP', 'MC-TOED - JOB ORDER', 'MC-TOED - PERMANENT', 'MHO - CASUAL', 'MHO - DOH', 'MHO - DOH-HRH', 'MHO - JOB ORDER', 'MHO - PERMANENT', 'OM - CASUAL', 'SICTO - GIP', 'SICTO - JOB ORDER', 'SICTO - PERMANENT']
